# Final Forecast

This notebook runs the final selected model on the groundwater time series project. It focuses on one model only : a linear autoregressive model using the data from the last 18 months, the month of the year (one-hot encoding), and summary statistics (average of the last 3, 6 and 12 month and the standard deviation of the last 12 month).

The data is preprocessed by interpolating the missing values (there are several months without any data) and then resampling everything to a monthly frequency.

The model is called autoregressive because it predicts several months into the future by using its own predictions as input data : To predict the month n+2, it will predict the month n+1 and append the result into the input.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# Number of months to use as input for the model
INPUT_LENGTH = 18

# Number of months to use for validation (the last 4 years of the series)
# This means that we will train our model on all the data except the last 48 months, and then we will evaluate our model on these 48 months.
VALID_MONTHS = 48


def load_monthly_series(csv_path):
    df = pd.read_csv(csv_path, sep=';', parse_dates=['date'], index_col='date')
    # Interpolate missing values and resample to monthly frequency, taking the mean of each month
    target = df['target'].interpolate(method='time')
    return target.resample('ME').mean()


def month_one_hot(date):
    # Example : for January, the output will be [1.0, 0.0, 0.0, ..., 0.0]
    values = [0.0] * 12
    values[date.month - 1] = 1.0
    return values


def make_features(history, next_date, input_length=INPUT_LENGTH):
    # Features are :
    # - The last `input_length` values of the series
    # - The month of the next date (one-hot encoded)
    # - The mean of the last 3, 6 and 12 values
    # - The standard deviation of the last 12 values
    values = np.asarray(history, dtype=float)
    features = values[-input_length:].tolist()
    features += month_one_hot(next_date)
    features += [
        values[-3:].mean(),
        values[-6:].mean(),
        values[-12:].mean(),
        values[-12:].std(),
    ]
    return np.asarray(features, dtype=float)


def build_training_data(series, input_length=INPUT_LENGTH):
    xs = [] # X axis : the features we will use to predict the next value of the series
    ys = [] # Y axis : the next value of the series that we want to predict
    for i in range(input_length, len(series)):
        history = series.iloc[i - input_length:i].values
        xs.append(make_features(history, series.index[i], input_length))
        ys.append(series.iloc[i])
    return np.asarray(xs), np.asarray(ys)


def recursive_forecast(model, history, future_index, input_length=INPUT_LENGTH):
    values = list(history.values)
    predictions = []
    for date in future_index:
        x = make_features(values, date, input_length).reshape(1, -1)
        pred = float(model.predict(x)[0])
        predictions.append(pred)
        # Update history with the new prediction for the next iteration : this is what makes it a recursive forecast !
        values.append(pred)
    return np.asarray(predictions)


In [ ]:
raw_daily = pd.read_csv('train.csv', sep=';', parse_dates=['date'])
monthly_series = load_monthly_series('train.csv')
test_dates = pd.read_csv('test.csv', sep=';', parse_dates=['date'])['date']

train_series = monthly_series.iloc[:-VALID_MONTHS]
valid_series = monthly_series.iloc[-VALID_MONTHS:]

print('Daily rows:', len(raw_daily))
print('Number of monthly observations:', len(monthly_series))
print('Train months:', len(train_series))
print('Validation months:', len(valid_series))
print('Validation period:', valid_series.index.min().date(), 'to', valid_series.index.max().date())


## Data Overview

The first figure shows the raw daily sensor values. The second figure shows the monthly series used by the model.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=False)

axes[0].plot(raw_daily['date'], raw_daily['target'], color='blue', linewidth=1.0)
axes[0].set_title('Raw Daily Sensor Series')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Groundwater level')
axes[0].grid(alpha=0.3)

axes[1].plot(train_series.index, train_series.values, color='blue', label='Train', linewidth=1.8)
axes[1].plot(valid_series.index, valid_series.values, color='orange', label='Validation', linewidth=1.8)
axes[1].axvline(valid_series.index[0], color='black', linestyle='--', alpha=0.7)
axes[1].set_title('Monthly Series After Interpolation and Resampling')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Groundwater level')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.show()


In [ ]:
X_train, y_train = build_training_data(train_series)
# Scikit-learn does most of the work for us here !
model = LinearRegression()
# We just need to fit the model on our training data.
model.fit(X_train, y_train)

valid_predictions = recursive_forecast(model, train_series, valid_series.index)

metrics_df = pd.DataFrame(
    {
        'metric': ['MAE', 'RMSE_3M', 'RMSE_6M', 'RMSE_12M', 'RMSE_48M'],
        'value': [
            mean_absolute_error(valid_series.values, valid_predictions), # This is the overall error on the validation set
            mean_squared_error(valid_series.values[:3], valid_predictions[:3]) ** 0.5,
            mean_squared_error(valid_series.values[:6], valid_predictions[:6]) ** 0.5,
            mean_squared_error(valid_series.values[:12], valid_predictions[:12]) ** 0.5,
            mean_squared_error(valid_series.values, valid_predictions) ** 0.5,
        ],
    }
)

metrics_df


## Validation Forecast

The first plot shows the full validation set against the prediction. The second plot zooms into the first 12 months, which is useful because we would most likely want to use this model to predict the immediate future, instead of years ahead.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

# Plot of the full dataset
history = train_series.iloc[:]
axes[0].plot(history.index, history.values, label='Train history', color='blue', marker='.', linewidth=1.4)
axes[0].plot(valid_series.index, valid_series.values, label='Validation truth', color='green', marker='o', linewidth=1.8)
axes[0].plot(valid_series.index, valid_predictions, label='Forecast', color='red', linestyle='--', marker='x', linewidth=1.8)
axes[0].set_title('Validation Forecast Over the Full dataset')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Groundwater level')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot of the validation period only
validation = valid_series.iloc[:]
axes[1].plot(valid_series.index, valid_series.values, label='Validation truth', color='green', marker='o', linewidth=1.8)
axes[1].plot(valid_series.index, valid_predictions, label='Forecast', color='red', linestyle='--', marker='x', linewidth=1.8)
axes[1].set_title('Validation Forecast Over the 48-Month Block validation period')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Groundwater level')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Zoom on the first 12 months of the validation period, which are the most likely to be accurate and needed in the real world (since the error will accumulate over time in a recursive forecast)
zoom_idx = valid_series.index[:12]
axes[2].plot(zoom_idx, valid_series.values[:12], label='Validation truth', color='green', marker='o', linewidth=1.8)
axes[2].plot(zoom_idx, valid_predictions[:12], label='Forecast', color='red', linestyle='--', marker='x', linewidth=1.8)
axes[2].fill_between(zoom_idx, valid_series.values[:12], valid_predictions[:12], color='red', alpha=0.08)
axes[2].set_title('Zoom on the First 12 Validation Months (most likely to be accurate and needed in the real world)')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Groundwater level')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## Data analysis

We can see two odd results from the previous graphes, both having a relatively straightforward explanation.

### Inaccuracy on the first two years

We can see on the Validation Forecast Over the 48-Month Block validation period that our error is the highest at the start of the validation graph, which is odd given the recursive nature of our algorithm : our predictions should be less effective the more predicted data is inserted as input for the algorithm.

However, this is most likely a coincidence : we can see on the full, raw dataset that the data related to this period features the biggest dip in groundwater level of the whole dataset, followed with a huge spike in a short timeframe : this is probably a rather "extraordinary" climatic event that is obviously harder to predict.

### Decreased accuracy on the june to september period

The last graph that zooms on the first 12 validation months (and the previous one that features four year of data) showcases a lack of accuracy on a specific period of the year : summer. The rest of the year, we are way closer to the truth.

This is also pretty simple to explain : summer is a season that introduces a lot of variation in the specific data we are studying : groundwater levels. From one summer to another, temperature and precipitation can be vastly different and unpredictable, which can impact the water underground unpredictably. On the contrary, colder seasons are much less likely to impact the weather in unpredictable ways.


## Final Forecast Export

After validation, the model is retrained on the full monthly history and used to forecast the dates contained in `test.csv`. We use the full dataset for training here for the best results.


In [ ]:
full_X, full_y = build_training_data(monthly_series)
final_model = LinearRegression()
final_model.fit(full_X, full_y)

final_predictions = recursive_forecast(final_model, monthly_series, pd.DatetimeIndex(test_dates))
final_forecast_df = pd.DataFrame({'date': test_dates, 'target': final_predictions})
final_forecast_df.to_csv('final_forecast_predictions.csv', sep=';', index=False, float_format='%.4f')

final_forecast_df.head(12)


## Conclusion

The final selected model is a linear autoregressive regression using the last 18 months, the month of the year, and rolling summary statistics.

### Results on Validation

- RMSE over the full 48-month validation block: about **1.09**
- RMSE over the first 12 months: about **1.38**
- RMSE over the first 6 months: about **0.61**
- RMSE over the first 3 months: about **0.12**

### Why This Model Was Chosen

This model was kept because it gave the best performance on the RMSE metric while staying simple and stable on a relatively small monthly dataset. Other, more complex models were considered not ideal on the small set of data at our disposal.

Furthermore, it displayed remarkably stable results over a long 48-month period, especially considering the recursive nature of the algorithm.

This project shows that the best forecasting model was in fact the simplest one, according to the results we got. On this dataset, a basic linear autoregressive model generalized better than the more sophisticated alternatives, which makes it the safest final choice for both performance and extrapolation.

However, it's interesting to note that the further we go into the future, the more the error will accumulate due to the nature of the recursive linear regression. However, it's interesting to note that the seasonality of the data plays in our favor here : since the data is smooth, seasonal and periodic, the impact is lessened.
